# Bifurcation Hamiltonian — construction, fork graph, and spectrum

Companion notebook to `bifurcation_hamiltonian.md`. We add the Denby–Peterson
fork term to the segment Hamiltonian in two forms and look at the matrix, the
fork graph, the analytic small clusters, and how the spectrum changes.

- **off-diag:** $`A'=(\gamma+\delta)I-C+\beta B`$, $`\mathbf b=\delta\mathbf1`$
- **full:** $`A''=(\gamma+\delta+2\beta)I-C+\beta B`$, $`\mathbf b=(\delta+\beta)\mathbf1`$

$`C`$ = continuation adjacency (base off-diagonal); $`B`$ = fork adjacency
(segments sharing a start- or end-hit). γ=3, δ=1, s=γ+δ=4.

In [1]:
import sys
sys.path.insert(0, "/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Bifurification")
from pathlib import Path
import numpy as np, scipy.sparse as sp
import matplotlib.pyplot as plt
import bif
plt.rcParams.update({"figure.dpi":110,"font.size":11,"axes.grid":True,"grid.alpha":0.3})
OUT = Path(bif.__file__).resolve().parent/"outputs"; OUT.mkdir(parents=True, exist_ok=True)
G,D,S = bif.GAMMA, bif.DELTA, bif.GAMMA+bif.DELTA
print("gamma+delta =", S, "  base tau =", bif.threshold(0,'off'))

gamma+delta = 4.0   base tau = 0.35


## 1. Analytic small example — a real track with a false fork
A clean track chain $`s_1\!-\!s_2\!-\!s_3\!-\!s_4`$ plus a false segment $`f`$ that
shares the start-hit of $`s_1`$ (one fork edge $`B_{s_1 f}=1`$). In the base model
$`f`$ is isolated on the notch at $`x_f=0.25`$.

In [2]:
A0 = S*np.eye(5)
for i,j in [(0,1),(1,2),(2,3)]: A0[i,j]=A0[j,i]=-1
B = np.zeros((5,5)); B[0,4]=B[4,0]=1
def solve(A,b): return np.linalg.solve(A,b)
for mode in ['off','full']:
    print(f"\n--- {mode} ---   (true s1..s4 | false fork f | attractor)")
    for beta in [0.0,0.25,0.5,1.0,2.0]:
        if mode=='off': A=A0+beta*B; b=D*np.ones(5); attr=D/S
        else: A=A0+2*beta*np.eye(5)+beta*B; b=(D+beta)*np.ones(5); attr=(D+beta)/(S+2*beta)
        x=solve(A,b); w=np.linalg.eigvalsh(A)
        print(f"  beta={beta:4.2f}: {np.round(x[:4],3)}  f={x[4]:6.3f}  attr={attr:.3f}  eig={np.round(w,3)}")
print("\nIsolated fork: x_f = (delta - beta*x_s1)/(gamma+delta) -> driven below 0.25 toward 0 (off form).")


--- off ---   (true s1..s4 | false fork f | attractor)
  beta=0.00: [0.364 0.455 0.455 0.364]  f= 0.250  attr=0.250  eig=[2.382 3.382 4.    4.618 5.618]
  beta=0.25: [0.348 0.45  0.453 0.363]  f= 0.228  attr=0.250  eig=[2.377 3.347 4.    4.653 5.623]
  beta=0.50: [0.336 0.447 0.453 0.363]  f= 0.208  attr=0.250  eig=[2.359 3.254 4.    4.746 5.641]
  beta=1.00: [0.318 0.442 0.451 0.363]  f= 0.171  attr=0.250  eig=[2.268 3.    4.    5.    5.732]
  beta=2.00: [0.314 0.441 0.451 0.363]  f= 0.093  attr=0.250  eig=[1.697 2.697 4.    5.303 6.303]

--- full ---   (true s1..s4 | false fork f | attractor)
  beta=0.00: [0.364 0.455 0.455 0.364]  f= 0.250  attr=0.250  eig=[2.382 3.382 4.    4.618 5.618]
  beta=0.25: [0.366 0.463 0.465 0.381]  f= 0.257  attr=0.278  eig=[2.877 3.847 4.5   5.153 6.123]
  beta=0.50: [0.367 0.468 0.472 0.394]  f= 0.263  attr=0.300  eig=[3.359 4.254 5.    5.746 6.641]
  beta=1.00: [0.367 0.475 0.481 0.414]  f= 0.272  attr=0.333  eig=[4.268 5.    6.    7.    7.732]
  bet

## 2. On a real event — the fork graph is dense
$`C`$ is sparse ($`O(T^2)`$). $`B`$ is dense: every interior hit starts $`O(T)`$
segments, all mutually forked, so $`\mathrm{nnz}(B)=O(T^3)`$. This is what makes the
quantum (1BQF) solve expensive (next notebook).

In [3]:
rows=[]
for T in [10,20,50,100]:
    ev=bif.event(T); ham=bif.base_hamiltonian(ev); A0=ham.A.tocsr()
    Bm=bif.fork_graph(ham._segment_to_hit_ids); truth=bif.truth_mask(ev)
    deg=np.asarray(Bm.sum(1)).ravel()
    rows.append(dict(T=T, n_seg=ham.n_segments, nnz_C=A0.nnz, nnz_B=int(Bm.nnz),
                     forkdeg_true=float(np.median(deg[truth])), forkdeg_false=float(np.median(deg[~truth]))))
import pandas as pd; inv=pd.DataFrame(rows); display(inv)
Tv=inv["T"].to_numpy()
fig,ax=plt.subplots(figsize=(7,4.6))
ax.loglog(Tv, inv.nnz_C, 'o-', label="nnz(C) continuation ~ $T^2$")
ax.loglog(Tv, inv.nnz_B, 's-', label="nnz(B) fork ~ $T^3$")
ax.loglog(Tv, 4*Tv**2, 'k:', alpha=.5, label="$4T^2$"); ax.loglog(Tv, 2*Tv**3, 'r:', alpha=.5, label="$2T^3$")
ax.set_xlabel("T (tracks)"); ax.set_ylabel("non-zeros"); ax.legend(fontsize=9)
ax.set_title("Fork graph breaks the sparse-A invariant (true & false have equal fork degree)", fontweight="bold")
fig.tight_layout();
for e,dp in (("pdf",600),("png",300)): fig.savefig(OUT/f"fork_graph_density.{e}",dpi=dp,bbox_inches="tight",facecolor="white")
plt.show(); print("saved fork_graph_density")

,T,n_seg,nnz_C,nnz_B,forkdeg_true,forkdeg_false
0,10,400,460,7200,18.0,18.0
1,20,1600,1720,60800,38.0,38.0
2,50,10000,10312,980000,98.0,98.0
3,100,40000,40624,7920000,198.0,198.0


saved fork_graph_density


## 3. The spectrum barely moves — the *solution* down-scales
Adding $`\beta B`$ couples the previously-isolated false segments, but a repulsive
fork star $`K_{1,m}`$ keeps $`m-1`$ eigenvalues **on** the notch (its null space), so
**most of the notch degeneracy survives**; only the symmetric fork combinations
split off. The real effect is on the *solution* scale, not the spectrum. (T=20.)

In [4]:
T=20; ev=bif.event(T); ham=bif.base_hamiltonian(ev); A0=ham.A.tocsr()
Bm=bif.fork_graph(ham._segment_to_hit_ids); truth=bif.truth_mask(ev)
fig,ax=plt.subplots(1,2,figsize=(13,4.8))
for beta,col in [(0.0,"#999999"),(0.05,"#4575b4"),(0.1,"#d73027")]:
    A,b,diag,tau=bif.bif_system(A0,Bm,beta,'off')
    w=np.linalg.eigvalsh(A.toarray())
    ax[0].hist(w,bins=120,histtype="step",lw=1.8,color=col,label=f"β={beta}")
ax[0].axvline(S,color="k",ls="--",lw=1.2,label="notch λ=γ+δ")
ax[0].set_yscale("log"); ax[0].set_xlabel("eigenvalue λ (off-diag form)"); ax[0].set_ylabel("count")
ax[0].set_title("(a) most of the notch survives (fork null space)", fontweight="bold"); ax[0].legend(fontsize=8)
# (b) uniform down-scaling + AUC preserved
betas=np.linspace(0,0.1,11); medT=[]; medF=[]; aucs=[]
for beta in betas:
    A,b,diag,tau=bif.bif_system(A0,Bm,beta,'off'); sol=bif.solve_classical(A,b)
    medT.append(np.median(sol[truth])); medF.append(np.median(sol[~truth])); aucs.append(bif.auc(sol,truth))
ax[1].plot(betas,medT,'o-',color="#1b7837",label="median true")
ax[1].plot(betas,medF,'s-',color="#c51b7d",label="median false")
ax[1].plot(betas,aucs,'^--',color="#2166ac",label="AUC(true:false)")
ax[1].axhline(bif.threshold(0,'off'),color="k",ls=":",lw=1,label="τ=0.35")
ax[1].set_xlabel("β (off-diag form)"); ax[1].set_ylabel("activation / AUC")
ax[1].set_title("(b) β down-scales uniformly; AUC stays ~1", fontweight="bold"); ax[1].legend(fontsize=8)
fig.tight_layout()
for e,dp in (("pdf",600),("png",300)): fig.savefig(OUT/f"spectrum_and_downscale.{e}",dpi=dp,bbox_inches="tight",facecolor="white")
plt.show(); print("saved spectrum_and_downscale")

saved spectrum_and_downscale


**Reading it.** (a) The base spectrum has a huge spike at the notch $`\lambda=\gamma+\delta`$
(the isolated false bulk). $`\beta B`$ couples those segments, but the fork graph has
a large null space (a repulsive star $`K_{1,m}`$ leaves $`m-1`$ modes on the notch),
so the spike **largely persists** — only a few symmetric modes split off. (b) The
visible effect is on the *solution*: true and false carry **equal fork degree**, so
the activations **down-scale** with $`\beta`$ (median true and median false fall
together) and the ranking (AUC ≈ 1) is preserved. At a *fixed* τ=0.35 this looks
like segments switching off — a threshold artefact, not a loss of separability
(quantified in notebook 02).